In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
pd.options.display.max_rows =20
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

In [ ]:
df = pd.read_parquet("daily-power-generation.parquet")

In [ ]:
df['date'] = pd.to_datetime(df['date'],format='mixed')
for col in ['region', 'state_name', 'sector', 'station_type']:
    df[col] = df[col].astype('category')

In [ ]:
df.info(verbose=True,show_counts=True)

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
# there was certain bad data from compilation, so cleaning it.
df = df[~df['power_station'].str.contains('Type|Total|TOTAL|Sum', case=False, na=False)]

### Data Distribution

Analytical Framework: Quantifying the absolute contribution of individual generation technologies to map the foundational energy mix.


Empirical Findings: Thermal power dominantly anchors the baseline at 83% of aggregate generation. Hydro infrastructure serves as the primary alternative at 10.8%, while Nuclear units maintain a rigid baseline share of 3.32%.


In [ ]:
sector_pie = df.groupby('station_type')['todays_gen_act'].sum().reset_index()
sector_pie_fig=px.pie(data_frame=sector_pie,names='station_type',values='todays_gen_act',title='total electricty generation by station type',)
sector_pie_fig.show()

### Grid Reliability & Fulfillment Metric

Analytical Framework: Evaluating cross-regional operational execution by benchmarking aggregate actual generation against programmed targets and assessing daily variance.


Empirical Findings: The Western and Eastern regions operate closest to programmed baselines, with the Eastern grid maintaining a slight surplus (101.26%) to offset wider system stress. The Southern region exhibits the most significant supply strain, running a persistent deficit with a 91.6% fulfillment rate.

In [ ]:
df['variance'] = df['todays_gen_act'] - df['todays_gen_prgm']

In [ ]:
regional_perf = df.groupby('region').agg(total_programmed=('todays_gen_prgm', 'sum'),total_actual=('todays_gen_act', 'sum'), avg_daily_variance=('variance', 'mean')).reset_index()

regional_perf['fulfillment_rate'] = (regional_perf['total_actual'] / regional_perf['total_programmed']) * 100


In [ ]:
regional_perf

In [ ]:
# regional Performance plotting
labels = regional_perf['fulfillment_rate'].round(2).astype(str) + '%'
regional_perf_fig = px.bar(data_frame=regional_perf,x='region',y=['fulfillment_rate'],text=labels,)
regional_perf_fig.show()

### Asset Efficiency: Plant Capacity Utilization Rate
Analytical Framework: Calculating the Plant Load Factor (PLF) across asset classes by converting monitored capacity into maximum daily potential and measuring real-world output ratios.

Empirical Findings: Nuclear assets register the highest utilization efficiency, averaging a 69.65% operating rate. Thermal plants follow at 58.15%, whereas demand-driven or intermittent resources like Hydro (37.26%) and Gas-fired units (21.64%) reflect expectedly lower capacity deployment.

In [ ]:
active_plants = df[df['monitored_capacity']>0].copy()
active_plants['max_daily_gen_mu'] = (active_plants['monitored_capacity']* 24)/1000
active_plants['utilization_rate_pct'] = (active_plants['todays_gen_act']/active_plants['max_daily_gen_mu']) * 100
station_summary = active_plants.groupby('station_type')['utilization_rate_pct'].mean().sort_values(ascending=False).reset_index()
station_summary

In [ ]:
#plotting Station summary
labels = station_summary['utilization_rate_pct'].round(2).astype(str)+'%'
station_summary_fig=px.bar(data_frame=station_summary,x='station_type',y='utilization_rate_pct',text=labels)
station_summary_fig.update_layout(xaxis_title='Types of Power Stations',yaxis_title='Utilization rate in %')
station_summary_fig.show()

### Grid Risk Profiling: Chronic Outage Identification

Analytical Framework: Isolating system vulnerabilities and asset failure frequencies by filtering for days where programmed targets were active but actual production dropped to absolute zero.

Empirical Findings: High-volume zero-generation windows expose severe reliability risks in state-sector assets. Suratgarh TPS (8,484 outages) and Raichur TPS (7,166 outages) serve as the primary drivers of localized grid instability.

In [ ]:
outage_threshold = 0.5 
outages=df[(df['todays_gen_prgm']>outage_threshold)&(df['todays_gen_act']==0)]

frequent_failures = outages.groupby(['region', 'state_name', 'state_code', 'sector','station_type', 'power_station']).size().sort_values(ascending=False).reset_index(name='outages')


In [ ]:
print('power stations with frequent outages :\n',frequent_failures.head(10))

In [ ]:
#plotting frequent failures
frequent_failures_fig=px.treemap(data_frame=frequent_failures,path=['station_type','power_station'],values ='outages',height=800,width=800,
                                 title='Power Station Outages')
frequent_failures_fig.show()

### Power Generation Timeline Analysis:

Analytical Framework: Evaluating multi-year temporal trends from 2017 through 2025 to observe structural capacity shifts and seasonal baseline behavior.

Empirical Findings: Long-term trendlines confirm that Thermal generation absorbs the brunt of macroscopic demand spikes. The timeline clearly segregates the highly volatile, weather-dependent cycles of Hydro power from the flat, unvarying baseline output maintained by Nuclear infrastructure.

In [ ]:
timeline = df.groupby(['date', 'station_type'])['todays_gen_act'].sum().reset_index()

timeline_pivot = timeline.pivot(index='date', columns='station_type', values='todays_gen_act').reset_index()

timeline_pivot['Total']= timeline_pivot['Hydro'] + timeline_pivot['Nuclear'] +timeline_pivot['Ther (Dg)'] +timeline_pivot['Ther (Gt)'] + timeline_pivot['Thermal']

In [ ]:
timeline_pivot.head()

In [ ]:
# plotting timeline Pivot
timeline_pivot_fig = px.line(data_frame=timeline_pivot,x='date',y=['Hydro','Nuclear','Ther (Dg)','Ther (Gt)','Thermal'])
timeline_pivot_fig.update_layout(yaxis_title='Million Units',xaxis_title='timeline')
timeline_pivot_fig.show()

In [ ]:
#plotting total energy generated
total_fig = px.line(data_frame=timeline_pivot,x='date',y='Total',title='Total Energy generated Timeline')
total_fig.update_layout(yaxis_title='Million Units',xaxis_title='timeline')
total_fig.show()

### State-Level Grid Supply Disparity Analysis

Analytical Framework: Aggregating net planning-to-production deltas at the state level to identify structural supply anomalies and regional over-reliance.

Empirical Findings: Systemic Deficits: Tamil Nadu and Gujarat exhibit the highest absolute shortfalls, where real-world generation regularly drops below targets due to fuel constraints or unplanned downtime.

Systemic Surpluses: Chhattisgarh and West Bengal consistently outpace scheduled programming, acting as vital export anchors that stabilize the national grid.

In [ ]:
state_summary = df.groupby('state_name').agg(
    total_prog=('todays_gen_prgm', 'sum'),
    total_act=('todays_gen_act', 'sum')
)
state_summary['net_gap_mu'] = state_summary['total_act'] - state_summary['total_prog']
state_summary['performance_index'] = (state_summary['total_act'] / state_summary['total_prog']) * 100

# Sort to find the top net surplus and net deficit states
underperforming_states = state_summary.sort_values(by='net_gap_mu').head(5)
overperforming_states = state_summary.sort_values(by='net_gap_mu', ascending=False).head(5)

In [ ]:
underperforming_states

In [ ]:
overperforming_states

In [ ]:
# plotting net gap in planning and production
state_summary_fig= px.bar(data_frame=state_summary,x=state_summary.index,y='net_gap_mu',height=600,)
state_summary_fig.show()

### Regional Generation Profiles: Cross-Tabulation Matrix

Analytical Framework: Generating a row-normalized percentage distribution matrix to isolate independent regional fuel dependencies irrespective of total national volume.

Empirical Findings: The matrix exposes severe geographic dependency imbalances. The Eastern and Western zones are strictly locked into thermal generation (exceeding 90%), whereas the North Eastern grid displays a diversified, lower-carbon portfolio anchored by gas (48.1%) and hydro (31.2%).

In [ ]:
# Pivot matrix to see energy mix percentages by region
region_mix = pd.crosstab(index=df['region'],columns=df['station_type'],values=df['todays_gen_act'],aggfunc='sum',normalize='index') * 100

region_mix

In [ ]:
# plotting corelation betn power production corelation betn region
region_mix_fig= px.imshow(region_mix,height=500,width=500,color_continuous_scale='Picnic',text_auto='.1f')
region_mix_fig.show()

### Dynamic Flexibility: Ramping Volatility Analysis

Analytical Framework: Measuring first-order temporal differences per generation unit to calculate the standard deviation of daily output shifts during active operational phases.

Empirical Findings: Nuclear and Thermal units exhibit the highest standard deviation in daily ramps. This proves that baseload assets face intense mechanical stress, swinging output constantly to balance the grid due to a deficit in utility-scale storage alternatives.

In [ ]:
df = df.sort_values(by=['power_station_unit', 'date'])

df['daily_ramp'] = df.groupby('power_station_unit')['todays_gen_act'].diff()

active_operating_days = df[(df['todays_gen_prgm']>0.5) & (df['todays_gen_act']!=0.0)]

true_active_volatility = active_operating_days.groupby('station_type')['daily_ramp'].std().sort_values(ascending=False).reset_index()
true_active_volatility

In [ ]:
true_active_volatility_fig =px.bar(data_frame=true_active_volatility,x='station_type',y='daily_ramp',text_auto=True,
                                  title='Active Daily Ramping Volatility by Station Type')
true_active_volatility_fig.update_layout(xaxis_title='Station Type',yaxis_title='Volatility(sd in MU)')
true_active_volatility_fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,7),dpi=120)
sns.violinplot(data=active_operating_days,x='station_type',y='daily_ramp')
sns.despine(right=True,left=True,bottom=True,top=True)
plt.title('Distribution of Daily Ramping Changes by Power plants', fontsize=14, pad=15)
plt.xlabel('Power Stations', fontsize=12)
plt.ylabel('Daily Generation Change (MU)', fontsize=12)
#plt.ylim(-100,50)
plt.minorticks_on()
plt.grid(which='major', linestyle=':', linewidth='0.5', color='grey',axis='y')
plt.grid(which='minor', linestyle=':', linewidth='0.5', color='lightgrey',axis='y')

plt.show()

### Pareto Allocation of Grid Deficits
Analytical Framework: Modeling a cumulative percentage distribution of negative variance days across facilities to isolate the primary drivers of total supply shortfalls.

Empirical Findings: The distribution reflects a classic Pareto structure. The top 50% of the entire grid's supply deficit is heavily concentrated within a tiny cluster of high-capacity assets, led directly by massive merchant operations like Mundra UMTPP and North Chennai TPS.

In [ ]:
# Isolate deficit days
deficits = df[df['variance'] < 0].copy()
deficits['absolute_deficit'] = deficits['variance'].abs()

# Group by power station and find total lost energy
station_deficits = deficits.groupby('power_station')['absolute_deficit'].sum().sort_values(ascending=False).reset_index()

# Calculate running cumulative percentage
total_grid_deficit = station_deficits['absolute_deficit'].sum()
station_deficits['cumulative_pct'] = (station_deficits['absolute_deficit'].cumsum()/total_grid_deficit) * 100

# Filter for the stations causing the top 50% of all grid shortfalls
chronic_offenders = station_deficits[station_deficits['cumulative_pct'] <= 50].reset_index()

In [ ]:
chronic_offenders_fig=px.area(data_frame=chronic_offenders,x='power_station',y='cumulative_pct',hover_data='absolute_deficit',height=500)
chronic_offenders_fig.update_layout(xaxis_title='Power Station',yaxis_title='% of grid shortfall')
chronic_offenders_fig.show()

### Regional Interdependence and Deficit Balancing Analysis

Analytical Framework: Deriving a Pearson correlation matrix from daily regional generation variances to evaluate inter-regional balancing and power transfer capabilities.

Empirical Findings: Low correlation coefficients between most zones indicate independent risk profiles, allowing power to be routed dynamically to cover localized deficits. However, positive correlations between the Southern, Northern, and Western grids point to synchronized risk windows where concurrent demand spikes limit mutual backup capacity.

In [ ]:
daily_regional_variance = df.groupby(['date', 'region'])['variance'].sum().unstack()
grid_correlation = daily_regional_variance.corr()
print(grid_correlation)

In [ ]:
a = px.imshow(grid_correlation,height=600,width=600)
a.show()